# Báo cáo Thực hành: Đánh giá, Suy luận và Đo lường độ trễ mô hình SVTRv2 (OpenOCR)
**Học viên thực hiện:** TẠ CÔNG NGHĨA

**Mã số học viên:** 250101048

**Chuyên ngành:** Khoa học Máy tính

---

## 📌 Mục tiêu
Notebook này được xây dựng nhằm mục đích cài đặt môi trường và thực nghiệm các quy trình cốt lõi của kiến trúc **SVTRv2** bao gồm:
1. Đánh giá hiệu suất mô hình (Evaluation) trên các tập văn bản ngắn và dài.
2. Kiểm thử suy luận (Inference) trên ảnh thực tế.
3. Đo lường tốc độ xử lý/độ trễ (Latency Measurement).

## 1. Thiết lập môi trường (Environment Setup)
Quá trình này sẽ clone mã nguồn từ nhánh `develop` của kho lưu trữ `OpenOCR` và cài đặt các phụ thuộc cần thiết.
*(Lưu ý: Môi trường dưới đây được cấu hình tối ưu cho PyTorch 2.2.0 và CUDA 11.8)*

In [20]:
# Clone mã nguồn
# !git clone -b main https://github.com/Topdu/OpenOCR.git

# Di chuyển vào thư mục dự án
%cd OpenOCR

# Cài đặt PyTorch, NumPy 1.x (cần thiết để tránh lỗi NumPy 2.0) và các thư viện liên quan
!pip install -r requirements.txt
!python3 build_package.py

[Errno 2] No such file or directory: 'OpenOCR'
/workspace/OpenOCR
OpenOCR Package Build Script

Step 1: Creating package structure...
Root directory: /workspace/OpenOCR
Target openocr directory: /workspace/OpenOCR/build/openocr
Removing existing /workspace/OpenOCR/build/openocr...
Creating /workspace/OpenOCR/build/openocr...
Copying configs...
Copying docs...
Copying opendet...
Copying openrec...
Copying tools...
Copying __init__.py...
Copying openocr.py...
Copying demo_gradio.py...
Copying demo_opendoc.py...
Copying demo_unirec.py...
Copying setup.py...
Copying MANIFEST.in...
Copying LICENSE...
Copying pyproject.toml...
Copying QUICKSTART.md...
Package structure created successfully!

Step 2: Building wheel package...


Cleaning /workspace/OpenOCR/build/dist...
Cleaning /workspace/OpenOCR/build/build...
Cleaning /workspace/OpenOCR/build/openocr_python.egg-info...

Running: python setup.py sdist bdist_wheel
running sdist
running egg_info
creating openocr_python.egg-info
writing openocr

## 2. Đánh giá mô hình (Evaluation)
Quá trình đánh giá được chia làm hai phân hệ: văn bản ngắn (Common, Union14M-Benchmark, OST) và văn bản dài (LTB). Kết quả đầu ra sẽ được lưu tự động dưới định dạng `.csv` tại thư mục `output_dir` khai báo trong file cấu hình.

*Lưu ý: Bạn cần đảm bảo đã tải pre-trained weights từ Google Drive của repo và đặt đúng vào đường dẫn yêu cầu.*

In [22]:
# !pip install "numpy<2.0.0" opencv-python tifffile scikit-image imgaug
# 2.1. Đánh giá trên tập dữ liệu văn bản ngắn
print("Đang khởi chạy tiến trình đánh giá trên tập dữ liệu ngắn...")
!/venv/main/bin/python tools/eval_rec_all_en.py \
    --c configs/rec/svtrv2/svtrv2_vn.yml

# 2.2. Đánh giá trên tập dữ liệu văn bản dài (LTB)
print("\nĐang khởi chạy tiến trình đánh giá trên tập LTB...")
!/venv/main/bin/python tools/eval_rec_all_long.py \
    --c configs/rec/svtrv2/svtrv2_vn.yml \
    --o Eval.loader.max_ratio=20

Đang khởi chạy tiến trình đánh giá trên tập dữ liệu ngắn...
[2026/07/01 07:30:57] openrec INFO: ----------- Config -----------
[2026/07/01 07:30:57] openrec INFO: Architecture : 
[2026/07/01 07:30:57] openrec INFO:     Decoder : 
[2026/07/01 07:30:57] openrec INFO:         ctc_decoder : 
[2026/07/01 07:30:57] openrec INFO:             name : RCTCDecoder
[2026/07/01 07:30:57] openrec INFO:         detach : False
[2026/07/01 07:30:57] openrec INFO:         gtc_decoder : 
[2026/07/01 07:30:57] openrec INFO:             ds : True
[2026/07/01 07:30:57] openrec INFO:             max_len : 25
[2026/07/01 07:30:57] openrec INFO:             name : SMTRDecoder
[2026/07/01 07:30:57] openrec INFO:             next_mode : True
[2026/07/01 07:30:57] openrec INFO:             num_layer : 1
[2026/07/01 07:30:57] openrec INFO:             sub_str_len : 5
[2026/07/01 07:30:57] openrec INFO:         infer_gtc : True
[2026/07/01 07:30:57] openrec INFO:         name : GTCDecoder
[2026/07/01 07:30:57] open

## 3. Suy luận trực tiếp (Inference)
Sử dụng script `infer_rec.py` để trích xuất văn bản từ một hình ảnh cụ thể hoặc một thư mục hình ảnh.

In [23]:
import os

# Khai báo đường dẫn hình ảnh cần suy luận
# Bạn có thể thay đổi IMAGE_PATH sang đường dẫn hình ảnh thực tế trên máy/Drive của bạn
IMAGE_PATH = "./docs/imgs_vn/ve_so.png"

if os.path.exists(IMAGE_PATH):
    print(f"Bắt đầu suy luận cho hình ảnh: {IMAGE_PATH}")
    !/venv/main/bin/python tools/infer_rec.py \
        --c configs/rec/svtrv2/svtrv2_vn.yml \
        --o Global.infer_img={IMAGE_PATH}
else:
    print(f"Cảnh báo: Không tìm thấy hình ảnh tại {IMAGE_PATH}. Vui lòng kiểm tra lại.")

Bắt đầu suy luận cho hình ảnh: ./docs/imgs_vn/ve_so.png
[2026/07/01 07:31:53] openrec INFO: Loading weights using torch.load: ./svtrv2_smtr_gtc_rctc/best.pth
[2026/07/01 07:31:53] openrec INFO: finetune from checkpoint ./svtrv2_smtr_gtc_rctc/best.pth
[2026/07/01 07:31:54] openrec INFO: 0 ./docs/imgs_vn/ve_so.png	 result: VESSO	0.6799009442329407, time cost: 0.41775012016296387
[2026/07/01 07:31:54] openrec INFO: Results saved to ./rec_results/rec_results.txt.)
[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0.41775012016296387]
[2026/07/01 07:31:54] openrec INFO: Sample num: 1, Weighted Avg time cost: 0.41775012016296387, Avg time cost: 0.41775012016296387
[2026/07/01 07:31:54] openrec INFO: success!


In [26]:
# Đường dẫn chứa tập VinText
TEST_IMAGE_FOLDER = "./docs/imgs_vn/"

print("Đang tiến hành đo lường Latency trên tập VinText...")
!/venv/main/bin/python ./tools/infer_rec.py \
    --c configs/rec/svtrv2/svtrv2_vn.yml \
    --o Global.infer_img={TEST_IMAGE_FOLDER}

Đang tiến hành đo lường Latency trên tập VinText...
[2026/07/01 07:40:05] openrec INFO: Loading weights using torch.load: ./svtrv2_smtr_gtc_rctc/best.pth
[2026/07/01 07:40:05] openrec INFO: finetune from checkpoint ./svtrv2_smtr_gtc_rctc/best.pth
[2026/07/01 07:40:06] openrec INFO: 0 ./docs/imgs_vn/im1201.jpg	 result: THINH	0.24074725806713104, time cost: 0.418304443359375
[2026/07/01 07:40:06] openrec INFO: 1 ./docs/imgs_vn/im1508.jpg	 result: Voositonel	0.20258793234825134, time cost: 0.020765066146850586
[2026/07/01 07:40:06] openrec INFO: Results saved to ./rec_results/rec_results.txt.)
[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0.418304443359375, 0.020765066146850586]
[2026/07/01 07:40:06] openrec INFO: Sample num: 2, Weighted Avg time cost: 0.2195347547531128, Avg time cost: 0.2195347547531128
[2026/07/01 07:40:06] openrec INFO: success!
